In [1]:
import pandas as pd
from sqlalchemy import create_engine, String, text

# Connexion MySQL
engine = create_engine("mysql+pymysql://root@localhost/dfe")

# Étape 1 : créer la table pivot avec clé primaire et index
create_table_sql = """
CREATE TABLE IF NOT EXISTS arrangement_customer (
    arrangement_id VARCHAR(50) NOT NULL,
    customer_id VARCHAR(50) NOT NULL,
    PRIMARY KEY (arrangement_id, customer_id)
) ;
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    print("✅ Table arrangement_customer créée (ou déjà existante)")

# Étape 2 : Charger les données depuis aa_arrangement_mcbc_live_full
df = pd.read_sql(
    "SELECT id AS arrangement_id, customer FROM aa_arrangement_mcbc_live_full",
    engine
)

# Étape 3 : Split la colonne customer
rows = []
for _, row in df.iterrows():
    arrangement_id = row["arrangement_id"]
    customers = str(row["customer"]).split("|") if row["customer"] else []
    for cust in customers:
        if cust.strip():  # éviter les vides
            rows.append((arrangement_id, cust.strip()))

# Étape 4 : Créer DataFrame pivot
df_pivot = pd.DataFrame(rows, columns=["arrangement_id", "customer_id"])

# Étape 5 : Insérer les données dans MySQL (remplacer si nécessaire)
df_pivot.to_sql(
    "arrangement_customer",
    engine,
    if_exists="replace",  # remplace la table si elle existe
    index=False,
        dtype={
            "arrangement_id": String(50),
            "customer_id": String(50)
        }
    )

# Étape 6 : Confirmation du nombre de lignes insérées
with engine.connect() as conn:
    count = pd.read_sql("SELECT COUNT(*) AS cnt FROM arrangement_customer", engine)["cnt"].iloc[0]
    print(f"✅ Insertion terminée : {count} lignes insérées dans arrangement_customer")
    
# Étape 7 : Créer les index pour optimiser les jointures


# Étape 7 : Aperçu des 5 premières lignes
print("\nExemple des données insérées :")
print(df_pivot.head())


✅ Table arrangement_customer créée (ou déjà existante)
✅ Insertion terminée : 92771 lignes insérées dans arrangement_customer

Exemple des données insérées :
  arrangement_id customer_id
0   AA243283YMZ9     5335136
1   AA243283YNNX     5335139
2   AA243283YXDM     5335189
3   AA243283Z3HL     5335229
4   AA243284BPQJ     5337763


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql
import math

GLOBAL_DATE= "20250915"

engine = create_engine(
    "mysql+pymysql://root@localhost/dfe",connect_args={"connect_timeout": 10}
)

# Requête SQL mise à jour pour récupérer les informations nécessaires
sql_query = """
WITH resolved_customer AS (
    SELECT DISTINCT
        ac.arrangement_id,
        CASE 
            WHEN c.sector = 1000 THEN CONCAT(c.short_name, ' ', c.name_1)
            ELSE c.name_1
        END AS Nom_compte
    FROM arrangement_customer ac
    JOIN customer_mcbc_live_full c
      ON ac.customer_id = c.id
)


SELECT 
    arrangement.co_code AS Agence, 
    arrangement.customer AS code_client,
    arrangement.linked_appl_id AS Numero_compte,
    arrangement.product AS Produits,
    rc.Nom_compte,
    customer.street AS Adresse,
    customer.sms_1 AS Contact,
    customer.gender AS Titre,
    customer.industry,
    customer.target,
    customer.legal_id AS Identification_Personne,
    NULL AS taux_d_interet, -- remplacé les multiples CASE qui renvoyaient NULL
    arrangement.product_group AS Type_Produit,
    contract_balance.open_balance AS open_balance,
    contract_balance.debit_mvmt AS debit_mvmt,
    contract_balance.credit_mvmt AS credit_mvmt,
    account.opening_date AS Date_effet,
    TIMESTAMPDIFF(MONTH, arrangement.start_date, account_details.maturity_date) AS Durée_en_mois,
    DATEDIFF(account_details.maturity_date, arrangement.start_date) AS Durée_en_jours, 
    account_details.maturity_date AS date_echeance,
    customer.account_officer AS chargé_clientele,
    CASE
        WHEN customer.sector = 1000 THEN 'Particulier'
        ELSE 'Morale'
    END AS categorie,
    contract_balance.type_sysdate,
    interet.id AS id_comp_2

FROM 
    aa_arrangement_mcbc_live_full AS arrangement
INNER JOIN 
    aa_account_details_mcbc_live_full AS account_details
    ON account_details.id = arrangement.id
LEFT JOIN 
    customer_mcbc_live_full AS customer
    ON arrangement.customer = customer.id
LEFT JOIN 
    eb_cont_bal_mcbc_live_full AS contract_balance
    ON contract_balance.id = arrangement.linked_appl_id
LEFT JOIN 
    account_mcbc_live_full AS account
    ON account.id = arrangement.linked_appl_id 
LEFT JOIN 
    aa_arr_interest_mcbc_live_full AS interet
    ON SUBSTRING_INDEX(interet.id, '-', 1) = arrangement.id
LEFT JOIN 
    resolved_customer rc
    ON rc.arrangement_id = arrangement.id

WHERE 
    arrangement.product_line IN ('ACCOUNTS')
    AND arrangement.arr_status IN ('AUTH', 'CURRENT','PENDING.CLOSURE')
    AND arrangement.product_group IN ('DV.SP.MG') LIMIT 100 ;

"""
# Charger les données en utilisant SQLAlchemy avec Pandas
df = pd.read_sql(sql_query, engine)


# 1. Remplacer les NaN par '0' dans les colonnes concernées
for col in ['debit_mvmt', 'credit_mvmt', 'open_balance']:
    df[col] = df[col].fillna('0')

# 2. Traitement du code_client : ne garder que la première valeur avant '|'
df['code_client'] = df['code_client'].astype(str).apply(lambda x: x.split('|')[0] if '|' in x else x)

# 3. Affichage des premières lignes
print("Premiers résultats du DataFrame avec le code_client traité :")
print(df[['code_client', 'Numero_compte']].head())

# 4. Fonction inchangée mais légèrement nettoyée pour plus de clarté et performance
def extract_balance(row, date_limite=GLOBAL_DATE):
    montant_capital_total = 0.0

    # Vérifie la présence des colonnes nécessaires
    if not all(k in row for k in ['type_sysdate', 'debit_mvmt', 'credit_mvmt', 'open_balance']):
        return montant_capital_total

    if isinstance(row['type_sysdate'], str):
        type_sysdate_values = row['type_sysdate'].split('|')
        debit_values = row['debit_mvmt'].split('|')
        credit_values = row['credit_mvmt'].split('|')
        balance_values = row['open_balance'].split('|')

        for index, entry in enumerate(type_sysdate_values):
            # Filtrage basé sur CURACCOUNT et date limite
            if entry == "CURACCOUNT":
                is_valid = True
            elif entry.startswith("CURACCOUNT-"):
                date_part = entry.replace("CURACCOUNT-", "")
                is_valid = date_part <= date_limite
            else:
                is_valid = False

            if is_valid:
                debit = float(debit_values[index].strip() or '0') if index < len(debit_values) else 0.0
                credit = float(credit_values[index].strip() or '0') if index < len(credit_values) else 0.0
                balance = float(balance_values[index].strip() or '0') if index < len(balance_values) else 0.0

                montant_capital_total += debit + credit + balance

    return montant_capital_total


# 5. Application de la fonction ligne par ligne
df['montant_capital'] = df.apply(extract_balance, axis=1)

# 6. Ajout des colonnes debit / credit
df['debit'] = df['montant_capital'].apply(lambda x: x if x < 0 else 0)
df['credit'] = df['montant_capital'].apply(lambda x: x if x > 0 else 0)

# 7. Supprimer les doublons sur Numero_compte
df = df.drop_duplicates(subset=['Numero_compte'], keep='first')

# 8. Nettoyage final des colonnes inutiles
df.drop(columns=['type_sysdate', 'debit_mvmt', 'open_balance', 'credit_mvmt'], inplace=True)

# 9. Affichage des résultats
print("\nPremiers résultats du DataFrame avec montant_capital :")
print(df[['Numero_compte', 'montant_capital']].head())

# 10. Affichage du total
total_montant = df['montant_capital'].sum()
print(f"\nTotal des montants : {total_montant:.2f}")

# Exporter les résultats en CSV
df.to_excel('output/DAV.xlsx', index=False)

Premiers résultats du DataFrame avec le code_client traité :
  code_client Numero_compte
0     5327076   20000158845
1     5324989   20000138968
2     5310079   20000001148
3     5323366   20000124517
4     5323373   20000124584

Premiers résultats du DataFrame avec montant_capital :
  Numero_compte  montant_capital
0   20000158845        -76776.17
1   20000138968        -90208.57
2   20000001148       -143373.14
3   20000124517        -80771.08
4   20000124584       -209460.49

Total des montants : -6864855.33


OSError: Cannot save file into a non-existent directory: 'output'